**Análise de Qualidade e Eficiência no Atendimento ao Consumidor**

versão 1 - 01mar26

Análise exploratória

In [ ]:
import pandas as pd
import seaborn as srn
import numpy as np
import statistics as sts
import matplotlib.pyplot as plt
import duckdb

Carga da base

In [ ]:
# Specify the path to your CSV file in the Colab environment
csv_file_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/basecompleta2025-12.csv'

# Ensure the DuckDB connection 'con' is available. If not, re-establish it.
if 'con' not in locals():
    import duckdb
    con = duckdb.connect(database=':memory:', read_only=False)
    print("DuckDB connection re-established.")

table_name = "basecompleta_2025_12"

try:
    # Use DuckDB's read_csv_auto to automatically detect schema and separator, assuming semicolon
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_csv_auto('{csv_file_path}', sep=';')")
    print(f"Successfully loaded '{csv_file_path}' into DuckDB table '{table_name}'.")

    # Display the first 3 rows of the newly created table
    print(f"First 3 rows of '{table_name}':")
    display(con.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchdf())

except Exception as e:
    print(f"Error loading {csv_file_path} into DuckDB: {e}")

DuckDB connection re-established.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully loaded '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/basecompleta2025-12.csv' into DuckDB table 'basecompleta_2025_12'.
First 3 rows of 'basecompleta_2025_12':


,Gestor,Canal de Origem,Região,UF,Cidade,Sexo,Faixa Etária,Ano Abertura,Mês Abertura,Data Abertura,...,Assunto,Grupo Problema,Problema,Como Comprou Contratou,Procurou Empresa,Respondida,Situação,Avaliação Reclamação,Nota do Consumidor,Análise da Recusa
0,Programa Estadual de Proteção e Defesa do Cons...,Plataforma Web,SE,MG,Barroso,M,entre 41 a 50 anos,2025,10,2025-10-10,...,Crédito Pessoal e Demais Empréstimos (exceto f...,Cobrança / Contestação,Negativação indevida - desconhece motivo e/ou ...,Não comprei / contratei,N,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
1,Fundação de Proteção e Defesa do Consumidor,Plataforma Web,SE,SP,São Paulo,F,entre 41 a 50 anos,2025,10,2025-10-11,...,"Vestuário e Artigos de Uso Pessoal (roupa, cal...",Atendimento / SAC,Má qualidade no atendimento (descortesia / des...,Ganhei de presente,S,S,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
2,Secretaria Nacional do Consumidor,Plataforma Web,S,RS,Sant'Ana do Livramento,O,entre 21 a 30 anos,2025,10,2025-10-12,...,"Produtos relacionados a saúde, exceto medicame...",Contrato / Oferta,Recusa em cancelar compra/serviço no prazo de ...,Internet,S,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente


In [ ]:
df = con.execute(f"SELECT * FROM {table_name}").fetchdf()

print(f"Tabela '{table_name}' carregada para o DataFrame 'df'")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tabela 'basecompleta_2025_12' carregada para o DataFrame 'df'


Visualilzação inicial da base de dados

In [ ]:
df.shape

(317850, 30)

In [ ]:
df

,Gestor,Canal de Origem,Região,UF,Cidade,Sexo,Faixa Etária,Ano Abertura,Mês Abertura,Data Abertura,...,Assunto,Grupo Problema,Problema,Como Comprou Contratou,Procurou Empresa,Respondida,Situação,Avaliação Reclamação,Nota do Consumidor,Análise da Recusa
0,Programa Estadual de Proteção e Defesa do Cons...,Plataforma Web,SE,MG,Barroso,M,entre 41 a 50 anos,2025,10,2025-10-10,...,Crédito Pessoal e Demais Empréstimos (exceto f...,Cobrança / Contestação,Negativação indevida - desconhece motivo e/ou ...,Não comprei / contratei,N,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
1,Fundação de Proteção e Defesa do Consumidor,Plataforma Web,SE,SP,São Paulo,F,entre 41 a 50 anos,2025,10,2025-10-11,...,"Vestuário e Artigos de Uso Pessoal (roupa, cal...",Atendimento / SAC,Má qualidade no atendimento (descortesia / des...,Ganhei de presente,S,S,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
2,Secretaria Nacional do Consumidor,Plataforma Web,S,RS,Sant'Ana do Livramento,O,entre 21 a 30 anos,2025,10,2025-10-12,...,"Produtos relacionados a saúde, exceto medicame...",Contrato / Oferta,Recusa em cancelar compra/serviço no prazo de ...,Internet,S,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
3,Programa Estadual de Proteção e Defesa do Cons...,Plataforma Web,SE,MG,Juiz de Fora,F,mais de 70 anos,2025,10,2025-10-12,...,Conta corrente / Salário / Poupança /Conta Apo...,Vício de Qualidade,"Clonagem, fraude, furto e roubo",Telefone,S,S,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
4,Secretaria Nacional do Consumidor,Plataforma Web,NE,RN,S?o Gon?alo do Amarante,M,entre 41 a 50 anos,2025,10,2025-10-12,...,Telefonia Móvel Pré-paga,Cobrança / Contestação,Cobrança indevida / abusiva para alterar ou ca...,Telefone,S,S,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317845,Programa Estadual de Proteção e Defesa do Cons...,Plataforma Web,NE,CE,Russas,F,entre 31 a 40 anos,2025,12,2025-12-31,...,Energia Elétrica,Atendimento / SAC,Má qualidade no atendimento presencial ou outr...,Telefone,S,S,Finalizada avaliada,Resolvida,5,None
317846,Fundação de Proteção e Defesa do Consumidor,Plataforma Web,SE,SP,Suzano,M,entre 41 a 50 anos,2025,12,2025-12-31,...,Consulta / Monitoramento de CPF,Dados Pessoais e Privacidade,"Coleta, uso ou compartilhamento indevido ou nã...",Internet,S,S,Finalizada avaliada,Não Resolvida,1,None
317847,Secretaria de Estado da Justiça e Cidadania de...,Plataforma Web,S,SC,Palhoça,O,entre 31 a 40 anos,2025,12,2025-12-31,...,Consulta / Monitoramento de CPF,Dados Pessoais e Privacidade,"Coleta, uso ou compartilhamento indevido ou nã...",Internet,N,S,Finalizada avaliada,Não Resolvida,1,None
317848,Fundação de Proteção e Defesa do Consumidor,Plataforma Web,SE,SP,Ribeirão Pires,M,até 20 anos,2025,12,2025-12-31,...,Consulta / Monitoramento de CPF,Cobrança / Contestação,Cobrança indevida / abusiva para alterar ou ca...,Não comprei / contratei,N,S,Finalizada avaliada,Não Resolvida,3,None


Análise exploratória

In [ ]:
# Visualização de valores nulos
df.isnull().sum()

,0
Gestor,0
Canal de Origem,0
Região,0
UF,0
Cidade,0
Sexo,7
Faixa Etária,0
Ano Abertura,0
Mês Abertura,0
Data Abertura,0


>
>
> **A análise de valores nulos revela:**
>
> * Sexo - Irrelevante para o objeto da análise
> * Data Resposta, Data Análise e Data Recusa - Valores nulos têm sentido para o negócio, pois são eventos ainda não ocorridos
> * Prazo Analise Gestor - Valores podem ser nulos conforme regras de negócio do dicionário de dados
> * Tempo Resposta - Valores podem ser nulos conforme regras de negócio do dicionário de dados
> * Avaliação Reclamação -Valores podem ser nulos conforme regras de negócio do dicionário de dados
> * Nota do Consumidor - Valores podem ser nulos conforme regras de negócio do dicionário de dados
> * Análise da Recusa - Valores podem ser nulos conforme regras de negócio do dicionário de dados
>
> Nota: supõe-se que os nulos da base, são efetivamente nulos por motivos do negócio e não por falhas na geração da base. Ou seja, presume-se a geração (.csv original) e publicação por parte da fonte feitas com sucesso, sem erros



Análise coluna a coluna

**1. Gestor**

In [ ]:
df['Gestor'].describe()

,Gestor
count,317850
unique,16
top,Fundação de Proteção e Defesa do Consumidor
freq,83525


In [ ]:
df['Gestor'].value_counts()

,count
Gestor,
Fundação de Proteção e Defesa do Consumidor,83525
Secretaria Nacional do Consumidor,63556
Programa Estadual de Proteção e Defesa do Consumidor,36743
Departamento Estadual de Proteção e Defesa do Consumidor,21603
Superintendência de Proteção e Defesa do Consumidor do Estado da Bahia,17885
Secretaria de Estado de Defesa do Consumidor,16404
Instituto Municipal de Proteção e Defesa do Consumidor,15155
Secretaria de Estado da Justiça e Cidadania de Santa Catarina - Departamento de Defesa do Consumidor,13706
Superintendência de Proteção e Defesa do Consumidor,11886


> "Gestor"
>
> Possui 16 valores distintos

**2. Canal de Origem**

In [ ]:
df['Canal de Origem'].value_counts()

,count
Canal de Origem,
Plataforma Web,317850


> "Canal de Origem"
>
> Redundante para a análise, pois todos os registros da base correspondem a esse canal: "Plataforma Web"

**3. Região**

In [ ]:
df['Região'].describe()

,Região
count,317850
unique,5
top,SE
freq,158149


In [ ]:
df['Região'].value_counts()

,count
Região,
SE,158149
NE,58845
S,50705
CO,32500
N,17651


> "Região"
>
> Corresponde às 5 regiões geográficas do Brasil, irrelevante para a análise

**4. UF**

In [ ]:
df['UF'].value_counts()

,count
UF,
SP,83546
MG,36736
RJ,31542
PR,21607
BA,17896
RS,15392
SC,13706
GO,11891
DF,10323


In [ ]:
df['UF'].describe()

,UF
count,317850
unique,27
top,SP
freq,83546


> "UF"
>
> Corresponde às 27 unidades da federação (estados) do Brasil, irrelevante para a análise

**5. Cidade**

In [ ]:
df['Cidade'].value_counts()

,count
Cidade,
São Paulo,28646
Rio de Janeiro,15147
Brasília,10323
Belo Horizonte,8980
Salvador,7169
...,...
Santa Isabel do Rio Negro,1
Parati,1
Ponte Alta do Norte,1


> "Cidade"
>
> Irrelevante para a análise
>
> Curiosidade: supondo unicidade dos valores da coluna, 5314 correspondem a 95% dos 5570 municípios (cidades) do Brasil

**6. Sexo**

In [ ]:
df['Sexo'].value_counts()

,count
Sexo,
M,172550
F,145055
O,238


> "Sexo"
>
> Ainda que o dicionário de dados registre apenas "M" e "F" como valores possíveis, presume-se que foi adotado "O" como padrão para casos em que a pessoa não desejou informar o sexo ou para casos em que não se identificava com os gêneros "M" ou "F"

**7. Faixa Etária**

In [ ]:
df['Faixa Etária'].value_counts()

,count
Faixa Etária,
entre 31 a 40 anos,100416
entre 21 a 30 anos,87853
entre 41 a 50 anos,66398
entre 51 a 60 anos,29654
entre 61 a 70 anos,17466
até 20 anos,9128
mais de 70 anos,6935


> "Faixa Etária"
>
> Ainda que bastante rico para possíveis análises, dado irrelevante para o foco do estudo

**8. Ano Abertura**

In [ ]:
df['Ano Abertura'].value_counts()

,count
Ano Abertura,
2025,317850


> "Ano Abertura"
>
> Dado redundante pois o .csv é referente a dez25

**9. Mês Abertura**

In [ ]:
df['Mês Abertura'].value_counts()

,count
Mês Abertura,
11,242480
12,54368
10,21002


> "Mês Abertura"
>
> Ilustra o fato de que cada .csv da fonte pode conter reclamações que foram abertas em meses anteriores, ainda estão tramitando, e pode aparecer em 2 ou mais meses
>
> Razão pela qual não foram acumulados vários meses para o estudo, que tornaria inviável a verificação de redundâncias, pois não existe identificador único para as reclamações

**10. Data Abertura**

In [ ]:
df['Data Abertura'].value_counts()

,count
Data Abertura,
2025-11-07,16897
2025-11-06,14447
2025-11-10,13269
2025-11-11,12782
2025-11-12,11589
...,...
2025-10-18,35
2025-12-31,24
2025-10-12,7


> "Data Abertura"
>
> Registra a data da abertura da reclamação na plataforma "consumidor.gov"

**11. Data Resposta**

In [ ]:
df['Data Resposta'].value_counts ()

,count
Data Resposta,
2025-11-11,14963
2025-11-14,14369
2025-11-12,13324
2025-11-13,12234
2025-12-04,11737
2025-11-10,11581
2025-12-05,11534
2025-11-07,11523
2025-11-19,10030


> "Data Resposta"
>
> Data da resposta da reclamação pela empresa

**12. Data Análise**

In [ ]:
df['Data Análise'].value_counts()

,count
Data Análise,
2025-12-16,2373
2025-12-15,2123
2025-12-10,1988
2025-12-17,1956
2025-12-18,1934
...,...
2025-10-27,2
2025-11-02,2
2025-11-01,2


> "Data Análise"
>
> Data em que o Gestor analise a recusa da empresa

**13. Data Recusa**

In [ ]:
df['Data Recusa'].value_counts()

,count
Data Recusa,
2025-12-04,2910
2025-12-05,2442
2025-12-03,2295
2025-11-28,2146
2025-11-27,2080
...,...
2025-10-17,8
2025-10-26,7
2025-11-02,6


> "Data Recusa"
>
> Data em que a empresa recusou a reclamação
>
> Nessa mesma data a recusa é enviada para análise do Gestor

**14. Data Finalização**

In [ ]:
df['Data Finalização'].value_counts()

,count
Data Finalização,
2025-12-10,18387
2025-12-08,16809
2025-12-09,16760
2025-12-03,15545
2025-12-15,15008
2025-12-31,14778
2025-12-07,14574
2025-12-22,12376
2025-12-23,12224


> "Data Finalização"
>
> Data de finalização da reclamação

**15. Prazo Resposta**

In [ ]:
df['Prazo Resposta'].value_counts()

,count
Prazo Resposta,
2025-11-22,15982
2025-11-21,13824
2025-11-25,12272
2025-11-26,11764
2025-11-19,10474
...,...
2026-01-11,524
2026-01-14,385
2025-11-09,96


> "Prazo Resposta"
>
> Data limite para resposta da empresa

**16. Prazo Análise Gestor**

In [ ]:
df['Prazo Analise Gestor'].value_counts()

,count
Prazo Analise Gestor,
20,7076
19,3679
1,2210
17,2040
18,1926
6,1833
15,1825
14,1783
11,1776


> "Prazo Análise Gestor"
>
> Número de dias que o Gestor levou para analisar a recusa
>
> É a diferença de tempo entre 'Data Análise' e 'Data Recusa'

**17. Tempo Resposta**

In [ ]:
df['Tempo Resposta'].value_counts()

,count
Tempo Resposta,
10,32858
9,28209
8,27001
1,23187
7,21268
4,17693
2,17601
6,17111
3,15607


> "Tempo Resposta"
>
> Número de dias que a empresa levou para responder
>
> É a diferença de dias entre 'Data Reposta" e 'Data Abertura'
>
> Se for o caso, desconsidera o tempo em que a reclamação tenha ficado em análise pelo Gestor

**18. Nome Fantasia**

In [ ]:
df['Nome Fantasia'].value_counts()

,count
Nome Fantasia,
Nubank,31983
Serasa Experian,22677
Banco do Brasil,14364
Banco Bradesco,11588
Banco Santander,10788
...,...
Geral Bet,1
Vale Presente,1
Cooperbombril,1


> "Nome Fantasia"
>
> Nome pelo qual a empresa é conhecida no mercado
>
> Está mais ligado à "marca" do que ao registro jurídico da empresa

**19. Segmento de Mercado**

In [ ]:
df['Segmento de Mercado'].value_counts()

,count
Segmento de Mercado,
"Bancos, Financeiras e Administradoras de Cartão",158118
"Operadoras de Telecomunicações (Telefonia, Internet, TV por assinatura)",24370
Bancos de Dados e Cadastros de Consumidores,23342
Empresas de Pagamento Eletrônico,14307
Comércio Eletrônico,10998
Transporte Aéreo,10658
Empresas de Intermediação de Serviços / Negócios,9574
Energia Elétrica,9452
Empresas de Recuperação de Crédito,8104


In [ ]:
df['Segmento de Mercado'].describe()

,Segmento de Mercado
count,317850
unique,44
top,"Bancos, Financeiras e Administradoras de Cartão"
freq,158118


> "Segmento de Mercado"
>
> Principal segmento de mercado no qual a empresa atua

**20. Área**

In [ ]:
df['Área'].value_counts()

,count
Área,
Serviços Financeiros,214091
Demais Serviços,20416
Telecomunicações,20269
Transportes,12503
Demais Produtos,12290
"Água, Energia, Gás",12287
Produtos de Telefonia e Informática,8291
Produtos Eletrodomésticos e Eletrônicos,7502
Saúde,3506


> "Área"
>
> Se refere à área do assunto da reclamação

**21. Assunto**

In [ ]:
df['Assunto'].value_counts()

,count
Assunto,
Cartão de Crédito / Cartão de Débito / Cartão de Loja,76742
Crédito Pessoal e Demais Empréstimos (exceto financiamento de imóveis e veículos),47726
Consulta / Monitoramento de CPF,16355
Aéreo,11503
Crédito Consignado / Cartão de Crédito Consignado / RMC (para beneficiários do INSS),11421
...,...
Aquaviário,3
Ensino Fundamental,2
Ensino Médio,1


> "Assunto"
>
> É o assunto objeto da reclamação

**22. Grupo Problema**

In [ ]:
df['Grupo Problema'].value_counts()

,count
Grupo Problema,
Cobrança / Contestação,190834
Contrato / Oferta,32572
Atendimento / SAC,31057
Vício de Qualidade,25461
Dados Pessoais e Privacidade,22873
Entrega do Produto,8784
Informação,4679
Saúde e Segurança,1590


> "Grupo Problema"
>
> Agrupamento de problemas

**23. Problema**

In [ ]:
df['Problema'].value_counts()

,count
Problema,
Renegociação / parcelamento de dívida,45130
"Cálculo de juros, saldo devedor (contestação, solicitação de histórico, dúvidas)",38777
Cobrança indevida / abusiva para alterar ou cancelar o contrato,26012
"Cobrança de tarifas, taxas, valores não previstos / não informados",13504
Cobrança por serviço/produto não contratado / não reconhecido / não solicitado,12906
...,...
Fechamento da instituição (descontinuidade do serviço),3
Não fornecimento de nota fiscal/recibo,3
Cobrança irregular de taxa de corretagem,1


> "Problema"
>
> Descrição do problema objeto da reclamação

**24. Como Comprou Contratou**

In [ ]:
df['Como Comprou Contratou'].value_counts()

,count
Como Comprou Contratou,
Internet,160541
Não comprei / contratei,58847
Loja física,48354
Telefone,38325
Domicílio,6028
SMS / Mensagem de texto,2623
Catálogo,1612
"Stand, feiras e eventos",873
Ganhei de presente,647


> "Como Comprou Contratou"
>
> Descrição do meio pelo qual o consumidor comprou ou contratou o produto ou serviço

**25. Procurou Empresa**

In [ ]:
df['Procurou Empresa'].value_counts()

,count
Procurou Empresa,
S,237163
N,80687


> "Procurou Empresa"
>
> Flag 'S' ou 'N' que indica se o consumidor procurou a empresa para resolver o problema

**26. Respondida**

In [ ]:
df['Respondida'].value_counts()

,count
Respondida,
S,265558
N,52292


> "Respondida"
>
> Flag 'S' ou 'N' que indica se a empresa respondeu ou não a reclamação

**27. Situação**

In [ ]:
df['Situação'].value_counts()

,count
Situação,
Finalizada não avaliada,209288
Finalizada avaliada,65765
Cancelada,32821
Encerrada,9976


> "Situação"
>
> Situação (status) da reclamação no sistema

**28. Avaliação Reclamação**

In [ ]:
df['Avaliação Reclamação'].value_counts()

,count
Avaliação Reclamação,
Não Avaliada,209288
Não Resolvida,39031
Resolvida,26734


> "Avaliação Reclamação"
>
> Classificação feita pelo consumidor sobre o desfecho da reclamação

**29. Nota do Consumidor**

In [ ]:
df['Nota do Consumidor'].value_counts()

,count
Nota do Consumidor,
1,34866
5,17423
3,5438
4,4653
2,3385


> "Nota do Consumidor"
>
> Nota de 1 a 5 dada pelo consumidor ao atendimento da empresa

**30. Análise da Recusa**

In [ ]:
df['Análise da Recusa'].value_counts()

,count
Análise da Recusa,
Procedente,32755
Encerrada,9976
Improcedente,3719


> "Análise da Recusa"
>
> Resultado da análise da recusa se o Gestor aceita (procedente), rejeita (improcedente) ou não analisa no prazo (encerrada)

**Indicadores**

1. Índice de resolução por segmento

In [ ]:
# Reclamações Finalizadas Resolvidas
reclamacoes_resolvidas = df[(df['Situação'] == 'Finalizada avaliada') & (df['Avaliação Reclamação'] == 'Resolvida')].shape[0]

# Reclamações Finalizadas Não Avaliadas
reclamacoes_nao_avaliadas = df[df['Situação'] == 'Finalizada não avaliada'].shape[0]

# Total de Reclamações Finalizadas
total_reclamacoes_finalizadas = df[(df['Situação'] == 'Finalizada avaliada') | (df['Situação'] == 'Finalizada não avaliada')].shape[0]

# Calculate the solution index
if total_reclamacoes_finalizadas > 0:
    indice_solucao = (reclamacoes_resolvidas + reclamacoes_nao_avaliadas) / total_reclamacoes_finalizadas
else:
    indice_solucao = np.nan # Divisão por zero

print(f"Reclamações Finalizadas Resolvidas: {reclamacoes_resolvidas}")
print(f"Reclamações Finalizadas Não Avaliadas: {reclamacoes_nao_avaliadas}")
print(f"Total de Reclamações Finalizadas: {total_reclamacoes_finalizadas}")
print(f"Índice de Solução: {indice_solucao:.2f}")

Reclamações Finalizadas Resolvidas: 26734
Reclamações Finalizadas Não Avaliadas: 209288
Total de Reclamações Finalizadas: 275053
Índice de Solução: 0.86


> O "Índice de Solução" geral é 86%

In [ ]:
def calculate_solution_index(segment_df):
    # Reclamações Finalizadas Resolvidas
    # Filtra as reclamações dentro deste segmento que foram 'Finalizada avaliada' E 'Resolvida'.
    reclamacoes_resolvidas = segment_df[(segment_df['Situação'] == 'Finalizada avaliada') & (segment_df['Avaliação Reclamação'] == 'Resolvida')].shape[0]

    # Reclamações Finalizadas Não Avaliadas
    # Filtra as reclamações dentro deste segmento que foram 'Finalizada não avaliada'.
    reclamacoes_nao_avaliadas = segment_df[segment_df['Situação'] == 'Finalizada não avaliada'].shape[0]

    # Total de Reclamações Finalizadas
    # Filtra as reclamações dentro deste segmento que foram 'Finalizada avaliada' OU 'Finalizada não avaliada'.
    total_reclamacoes_finalizadas = segment_df[((segment_df['Situação'] == 'Finalizada avaliada') | (segment_df['Situação'] == 'Finalizada não avaliada'))].shape[0]

    # Calculate the solution index
    # Aplica a fórmula do índice de solução. Se não houver reclamações finalizadas, o índice é 0.0 para evitar divisão por zero.
    if total_reclamacoes_finalizadas > 0:
        indice_solucao = (reclamacoes_resolvidas + reclamacoes_nao_avaliadas) / total_reclamacoes_finalizadas
    else:
        indice_solucao = 0.0  # Handle division by zero

    # Retorna os resultados como uma Series do pandas, onde cada chave é o nome da métrica e o valor é o resultado calculado.
    return pd.Series({
        'Reclamações Resolvidas': reclamacoes_resolvidas,
        'Reclamações Não Avaliadas': reclamacoes_nao_avaliadas,
        'Total Finalizadas': total_reclamacoes_finalizadas,
        'Índice de Solução': indice_solucao
    })

In [ ]:
# Agrupa o DataFrame `df` pela coluna 'Segmento de Mercado'.
# Para cada grupo (segmento), a função `calculate_solution_index` é aplicada.
# O resultado é um novo DataFrame `solution_index_by_segment` onde o índice são os nomes dos segmentos e as colunas são as métricas retornadas pela função.
solution_index_by_segment = df.groupby('Segmento de Mercado').apply(calculate_solution_index)

# Ordena o DataFrame resultante pelo 'Índice de Solução' em ordem decrescente (`ascending=False`).
# Em seguida, `.head(10)` seleciona apenas as 10 primeiras linhas, que correspondem aos 10 segmentos com os maiores índices de solução.
top_10_segments = solution_index_by_segment.sort_values(by='Índice de Solução', ascending=False).head(10)

print("Top 10 Market Segments by Solution Index:")
print(top_10_segments)

> Calcular o desempenho de resolução de reclamações para os segmentos a fim de identificar quais setores estão se destacando neste aspecto

In [ ]:
plt.figure(figsize=(12, 8))
srn.barplot(x=top_10_segments.index, y='Índice de Solução', data=top_10_segments, palette='viridis', hue=top_10_segments.index, legend=False)
plt.title('Top 10 Market Segments by Solution Index')
plt.xlabel('Market Segment')
plt.ylabel('Solution Index')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print("Bar chart showing Top 10 Market Segments by Solution Index displayed.")

In [ ]:
plt.figure(figsize=(15, 10)) # Adjust figure size for better readability with inverted axes
srn.barplot(x='Índice de Solução', y=solution_index_by_segment.index, data=solution_index_by_segment, palette='viridis', hue=solution_index_by_segment.index, legend=False)
plt.title('All Market Segments by Solution Index (Inverted Axes)')
plt.xlabel('Solution Index')
plt.ylabel('Market Segment')
plt.yticks(fontsize=8) # Keep y-tick labels horizontal for better readability
plt.xticks(rotation=0) # No rotation needed for x-axis
plt.tight_layout()
plt.show()
print("Bar chart showing all Market Segments by Solution Index with inverted axes displayed.")

> Os segmentos em geral tem o Índice de Solução concentrados ao redor da média 86%